In [9]:
import requests
import os
from dotenv import load_dotenv

load_dotenv()

True

In [10]:
# SEC EDGAR — no API key needed, just a descriptive User-Agent
HEADERS = {"User-Agent": os.getenv("SEC_USER_AGENT")}

In [11]:
def get_company_facts(ticker: str) -> dict:
    """
    Fetches all XBRL financial facts for a company from SEC EDGAR.
    Returns structured JSON with revenue, assets, EPS, etc.
    """
    # First, get the CIK number for the ticker
    search_url = f"https://efts.sec.gov/LATEST/search-index?q=%22{ticker}%22&dateRange=custom&startdt=2024-01-01&forms=10-K"
    
    # EDGAR has a company tickers lookup
    tickers_url = "https://www.sec.gov/files/company_tickers.json"
    resp = requests.get(tickers_url, headers=HEADERS)
    tickers = resp.json()
    
    # Find our company
    cik = None
    for entry in tickers.values():
        if entry["ticker"] == ticker.upper():
            cik = str(entry["cik_str"]).zfill(10)  # pad to 10 digits
            company_name = entry["title"]
            break
    
    if not cik:
        raise ValueError(f"Ticker {ticker} not found")
    
    print(f"Found: {company_name} | CIK: {cik}")
    
    # Fetch all XBRL facts for this company
    facts_url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"
    facts_resp = requests.get(facts_url, headers=HEADERS)
    return facts_resp.json()

In [ ]:
facts = get_company_facts("AAPL")

Found: Apple Inc. | CIK: 0000320193


IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [14]:
# Extract annual revenue (US-GAAP: Revenues)
revenues = facts["facts"]["us-gaap"].get("Revenues", {}).get("units", {}).get("USD", [])

print(revenues)

[{'start': '2015-09-27', 'end': '2016-09-24', 'val': 215639000000, 'accn': '0000320193-18-000145', 'fy': 2018, 'fp': 'FY', 'form': '10-K', 'filed': '2018-11-05', 'frame': 'CY2016'}, {'start': '2016-09-25', 'end': '2016-12-31', 'val': 78351000000, 'accn': '0000320193-18-000145', 'fy': 2018, 'fp': 'FY', 'form': '10-K', 'filed': '2018-11-05', 'frame': 'CY2016Q4'}, {'start': '2017-01-01', 'end': '2017-04-01', 'val': 52896000000, 'accn': '0000320193-18-000145', 'fy': 2018, 'fp': 'FY', 'form': '10-K', 'filed': '2018-11-05', 'frame': 'CY2017Q1'}, {'start': '2017-04-02', 'end': '2017-07-01', 'val': 45408000000, 'accn': '0000320193-18-000145', 'fy': 2018, 'fp': 'FY', 'form': '10-K', 'filed': '2018-11-05', 'frame': 'CY2017Q2'}, {'start': '2016-09-25', 'end': '2017-09-30', 'val': 229234000000, 'accn': '0000320193-18-000145', 'fy': 2018, 'fp': 'FY', 'form': '10-K', 'filed': '2018-11-05', 'frame': 'CY2017'}, {'start': '2017-07-02', 'end': '2017-09-30', 'val': 52579000000, 'accn': '0000320193-18-000

In [15]:
# Filter to annual 10-K filings only
annual = [r for r in revenues if r.get("form") == "10-K"]
print("\nApple Annual Revenue (10-K filings):")
for r in sorted(annual, key=lambda x: x["end"])[-5:]:  # last 5 years
    print(f"  {r['end']}: ${r['val']:,.0f}")


Apple Annual Revenue (10-K filings):
  2017-12-30: $88,293,000,000
  2018-03-31: $61,137,000,000
  2018-06-30: $53,265,000,000
  2018-09-29: $265,595,000,000
  2018-09-29: $62,900,000,000
